# 🌾 Wheat Plant Disease: CNN Model Fine-Tuning
**Assigned Task:** Model Fine-Tuning & Optimization (Sprint 2 - Part B)
**Author:** Sachin Choudhary (Team Lead, AI/ML)
**Dataset:** [`kushagra3204/wheat-plant-diseases`](https://www.kaggle.com/datasets/kushagra3204/wheat-plant-diseases)

---
### Workflow:
1. Install all dependencies first.
2. Load data with augmentation pipeline.
3. Set up Pretrained MobileNetV2 backbone (Transfer Learning).
4. **Phase 1 (Feature Extraction):** Train classification head with frozen backbone ($lr = 10^{-3}$).
5. **Phase 2 (Deep Fine-Tuning):** Unfreeze top 30 layers with low learning rate ($lr = 10^{-5}$) to adapt to subtle lesion patterns.
6. Plot combined two-phase progression curves.
7. Save final model as `finetuned_model.h5` and class indices as `classes.json`.

In [ ]:
# Step 1: Install dependencies
!pip install -q tensorflow numpy pandas matplotlib seaborn pillow
print('Dependencies installed successfully!')


In [ ]:
# Step 2: Import libraries & check GPU
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow Version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))


In [ ]:
# Step 3: Find dataset path automatically
def find_data_path():
    candidates = [
        '/kaggle/input/wheat-plant-diseases/data',
        '/kaggle/input/wheat-plant-diseases',
        '/kaggle/input/data',
        './data'
    ]
    for p in candidates:
        if os.path.exists(os.path.join(p, 'train')):
            return os.path.join(p, 'train'), os.path.join(p, 'valid')
    for root, dirs, files in os.walk('/kaggle/input' if os.path.exists('/kaggle/input') else '.'):
        if 'train' in dirs and 'valid' in dirs:
            return os.path.join(root, 'train'), os.path.join(root, 'valid')
    return None, None

TRAIN_DIR, VALID_DIR = find_data_path()
print('Train Path:', TRAIN_DIR)
print('Valid Path:', VALID_DIR)


In [ ]:
# Step 4: Data Augmentation & Generators
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print(f'Total Classes: {num_classes}')


In [ ]:
# Step 5: Build Pretrained MobileNetV2 Architecture
# Load MobileNetV2 without classification head
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model for Phase 1
base_model.trainable = False

# Attach custom classification head
finetuned_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Lambda(lambda x: tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

finetuned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

finetuned_model.summary()


In [ ]:
# Step 6: Phase 1 Training - Feature Extraction (5 Epochs)
print('Starting Phase 1 Training (Frozen Base)...')
history_p1 = finetuned_model.fit(
    train_generator,
    epochs=5,
    validation_data=valid_generator
)
print('Phase 1 Training Completed!')


In [ ]:
# Step 7: Phase 2 Training - Deep Fine-Tuning Top 30 Layers (7 Epochs)
print('Starting Phase 2 Deep Fine-Tuning...')
base_model.trainable = True

# Freeze all layers except the top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with delicate learning rate
finetuned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_p2 = finetuned_model.fit(
    train_generator,
    epochs=12,
    initial_epoch=5,
    validation_data=valid_generator
)
print('Phase 2 Fine-Tuning Completed!')


In [ ]:
# Step 8: Plot Combined Two-Phase Accuracy & Loss Progression Curves
acc = history_p1.history['accuracy'] + history_p2.history['accuracy']
val_acc = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
loss = history_p1.history['loss'] + history_p2.history['loss']
val_loss = history_p1.history['val_loss'] + history_p2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))

ax1.plot(acc, label='Train Acc', marker='o')
ax1.plot(val_acc, label='Val Acc', marker='s')
ax1.axvline(x=4, color='gray', linestyle='--', label='Start Fine-Tuning (Phase 2)')
ax1.set_title('Fine-Tuned MobileNetV2 - Accuracy Progression')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(loss, label='Train Loss', marker='o', color='red')
ax2.plot(val_loss, label='Val Loss', marker='s', color='orange')
ax2.axvline(x=4, color='gray', linestyle='--', label='Start Fine-Tuning (Phase 2)')
ax2.set_title('Fine-Tuned MobileNetV2 - Loss Progression')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Step 9: Evaluate Fine-Tuned Model Validation Metrics
ft_val_loss, ft_val_acc = finetuned_model.evaluate(valid_generator)
print(f'Fine-Tuned Validation Accuracy: {ft_val_acc * 100:.2f}%')
print(f'Fine-Tuned Validation Loss:     {ft_val_loss:.4f}')


In [ ]:
# Step 10: Save Fine-Tuned Model and Class Mappings
finetuned_model.save('finetuned_model.h5')

combined_history = {
    'accuracy': [float(x) for x in acc],
    'val_accuracy': [float(x) for x in val_acc],
    'loss': [float(x) for x in loss],
    'val_loss': [float(x) for x in val_loss]
}
with open('finetuned_history.json', 'w') as f:
    json.dump(combined_history, f)

with open('classes.json', 'w') as f:
    json.dump(valid_generator.class_indices, f, indent=4)

print('Saved finetuned_model.h5 successfully!')
print('Saved finetuned_history.json successfully!')
print('Saved classes.json successfully!')


### Summary of Fine-Tuning
- Pretrained MobileNetV2 fine-tuned with two-phase learning.
- Model saved as `finetuned_model.h5`.

👉 Now open `model_performance.ipynb` to evaluate and compare Baseline vs. Fine-Tuned model!